# **San Francisco's GTFS Dictionary**
**By: Ayemhenre Isikhuemhen**  
**February, 2026**

**Dataset:** SF MUNI General Transit Feed Specification (GTFS) — Current Feed  
**Source:** San Francisco Municipal Transportation Agency (SFMTA)  

**File Description:** 
This notebook explores the GTFS data files for San Francisco's MUNI transit system. GTFS (General Transit Feed Specification) is a standardized format for public transportation schedules and geographic information.  

## Dataset Overivew: File and Field Breakdown
---
Outlines the files that make up the dataset and explains the fields featured.

### San Francisco GTFS Data Files

| File | Records (approx) | Purpose |
|------|------------------|---------|
| `agency.txt` | 1 | Identifies the transit agency |
| `calendar.txt` | 4–10 | Defines weekly service patterns |
| `calendar_dates.txt` | 100–500 | Schedule exceptions (holidays, special events) |
| `directions.txt` | 100–200 | Human-readable direction labels |
| `fare_attributes.txt` | 3–8 | Fare products and prices |
| `fare_rules.txt` | 50–150 | Maps fares to routes |
| `routes.txt` | 80–120 | All transit routes/lines |
| `shapes.txt` | 500k–2M | Geographic path geometries |
| `stop_times.txt` | 2M–5M | Every arrival/departure |
| `stops.txt` | 5k–8k | All physical stops/stations |
| `timepoints.txt` | 20k–50k | Schedule adherence checkpoints |
| `trips.txt` |  30k–60k | Individual vehicle runs |


### agency.txt
**Purpose:** Identifies the transit agency providing the data.

| Field | Type | Required | Description | Example |
|-------|------|----------|-------------|---------|
| `agency_id` | ID | Optional | Unique identifier for the agency | `SFMTA` |
| `agency_name` | Text | **Required** | Full agency name | `San Francisco Municipal Transportation Agency` |
| `agency_url` | URL | **Required** | Agency website | `https://www.sfmta.com` |
| `agency_timezone` | Timezone | **Required** | Timezone for all times in the feed | `America/Los_Angeles` |
| `agency_lang` | Language code | Optional | Primary language | `en` |
| `agency_phone` | Phone | Optional | Voice telephone number | `311` |
| `agency_fare_url` | URL | Optional | URL for purchasing tickets | `https://www.sfmta.com/fares` |
| `agency_email` | Email | Optional | Customer service email | `customerservice@sfmta.com` |


### calendar.txt
**Purpose:** Defines service patterns using a weekly schedule (Monday–Sunday).

| Field | Type | Required | Description | Example |
|-------|------|----------|-------------|---------|
| `service_id` | ID | **Required** | Unique identifier for this service calendar | `WKDY` |
| `monday` | Enum (0/1) | **Required** | 1 = service runs on Mondays, 0 = does not | `1` |
| `tuesday` | Enum (0/1) | **Required** | 1 = service runs on Tuesdays | `1` |
| `wednesday` | Enum (0/1) | **Required** | 1 = service runs on Wednesdays | `1` |
| `thursday` | Enum (0/1) | **Required** | 1 = service runs on Thursdays | `1` |
| `friday` | Enum (0/1) | **Required** | 1 = service runs on Fridays | `1` |
| `saturday` | Enum (0/1) | **Required** | 1 = service runs on Saturdays | `0` |
| `sunday` | Enum (0/1) | **Required** | 1 = service runs on Sundays | `0` |
| `start_date` | Date (YYYYMMDD) | **Required** | First date this service is active | `20250101` |
| `end_date` | Date (YYYYMMDD) | **Required** | Last date this service is active | `20250630` |

**Common Service IDs in SF MUNI:**
- `WKDY`: Weekday service (Mon–Fri)
- `SAT`: Saturday service
- `SUN`: Sunday/holiday service


### calendar_dates.txt
**Purpose:** Exceptions to the regular calendar (holidays, special events, service disruptions).

| Field | Type | Required | Description | Example |
|-------|------|----------|-------------|---------|
| `service_id` | ID | **Required** | References `calendar.service_id` | `WKDY` |
| `date` | Date (YYYYMMDD) | **Required** | Date of the exception | `20250704` |
| `exception_type` | Enum | **Required** | `1` = service added, `2` = service removed | `2` |


### directions.txt
**Purpose:** *Non-standard extension.* Provides human-readable direction labels for route segments.

| Field | Type | Required | Description | Example |
|-------|------|----------|-------------|---------|
| `route_id` | ID | **Required** | References `routes.route_id` | `1` |
| `direction_id` | Enum (0/1) | **Required** | 0 = outbound, 1 = inbound (arbitrary) | `0` |
| `direction` | Text | **Required** | Human-readable direction label | `Outbound to Visitacion Valley` |

SFMTA uses this for rider-facing apps. Direction labels often reference terminus stops.  


### fare_attributes.txt
**Purpose:** Defines fare products, prices, and payment rules.

| Field | Type | Required | Description | Example |
|-------|------|----------|-------------|---------|
| `fare_id` | ID | **Required** | Unique identifier for this fare product | `adult` |
| `price` | Float | **Required** | Fare price (in `currency_type` units) | `2.50` |
| `currency_type` | Currency code | **Required** | ISO 4217 currency code | `USD` |
| `payment_method` | Enum | **Required** | `0` = on board, `1` = before boarding | `0` |
| `transfers` | Enum | **Required** | `0` = no transfers, `1` = one, `2` = two, blank = unlimited | `` |
| `transfer_duration` | Integer | Optional | Time window in seconds for valid transfers | `5400` |
| `agency_id` | ID | Conditionally Required | References `agency.agency_id` (if multiple agencies) | `SFMTA` |

**SF MUNI Fare Notes:**
- Adult cash fare: $2.50
- Transfers valid for 90 minutes (5400 seconds)
- Cable car has separate, higher fare
- Clipper card users get unlimited transfers within the window


### fare_rules.txt
**Purpose:** Maps fare products to specific routes or origin/destination zones.

| Field | Type | Required | Description | Example |
|-------|------|----------|-------------|---------|
| `fare_id` | ID | **Required** | References `fare_attributes.fare_id` | `adult` |
| `route_id` | ID | Optional | References `routes.route_id` (blank = all routes) | `1` |
| `origin_id` | ID | Optional | Origin zone ID | `` |
| `destination_id` | ID | Optional | Destination zone ID | `` |
| `contains_id` | ID | Optional | Zone that must be traversed | `` |

**Notes:**
- Most MUNI routes use the same base fare
- Cable cars are the primary exception
- SF doesn't use zone-based fares (all fields except `fare_id` and `route_id` are typically blank)


### routes.txt
**Purpose:** Defines all transit routes/lines in the system.

| Field | Type | Required | Description | Example |
|-------|------|----------|-------------|---------|
| `route_id` | ID | **Required** | Unique identifier for this route | `1` |
| `agency_id` | ID | Conditionally Required | References `agency.agency_id` | `SFMTA` |
| `route_short_name` | Text | Conditionally Required | Short public-facing name (e.g., "1", "N") | `1` |
| `route_long_name` | Text | Conditionally Required | Full descriptive name | `California` |
| `route_desc` | Text | Optional | Additional description | `Runs along California St` |
| `route_type` | Enum | **Required** | **See route type table below** | `3` |
| `route_url` | URL | Optional | Web page for this route | `https://www.sfmta.com/routes/1-california` |
| `route_color` | Hex color | Optional | Color (without #) for maps | `FF6B35` |
| `route_text_color` | Hex color | Optional | Text color for route badges | `FFFFFF` |
| `route_sort_order` | Integer | Optional | Display order in route lists | `10` |
  
**Route Types (GTFS Standard):**

| Code | Type | Description | MUNI Examples |
|------|------|-------------|---------------|
| `0` | Tram / Streetcar | Light rail | F line (historic streetcar) |
| `1` | Subway / Metro | Heavy rail | N, J, K, L, M, T (Muni Metro) |
| `2` | Rail | Intercity/long-distance | *(not used in SF)* |
| `3` | Bus | Regular bus service | 1, 5, 14, 38, etc. (majority of routes) |
| `4` | Ferry | Boat service | *(not operated by MUNI)* |
| `5` | Cable Car | Cable-drawn street car | Powell-Hyde, Powell-Mason, California St |
| `6` | Gondola | Aerial cable car | *(not in SF)* |
| `7` | Funicular | Steep incline rail | *(not in SF)* |


### shapes.txt
**Purpose:** Defines geographic path geometries for drawing route lines on maps.

| Field | Type | Required | Description | Example |
|-------|------|----------|-------------|---------|
| `shape_id` | ID | **Required** | Unique identifier for this shape | `1_0_var1` |
| `shape_pt_lat` | Latitude | **Required** | Latitude of shape point (WGS84) | `37.7912` |
| `shape_pt_lon` | Longitude | **Required** | Longitude of shape point (WGS84) | `-122.4104` |
| `shape_pt_sequence` | Integer | **Required** | Order of point in shape (starts at 1) | `1` |
| `shape_dist_traveled` | Float | Optional | Distance from first point (meters or feet) | `152.4` |


### stop_times.txt
**Purpose:** The **most granular** file — every scheduled arrival/departure for every trip.

| Field | Type | Required | Description | Example |
|-------|------|----------|-------------|---------|
| `trip_id` | ID | **Required** | References `trips.trip_id` | `1000001` |
| `arrival_time` | Time (HH:MM:SS) | **Required** | Arrival time at this stop | `08:45:30` |
| `departure_time` | Time (HH:MM:SS) | **Required** | Departure time from this stop | `08:45:30` |
| `stop_id` | ID | **Required** | References `stops.stop_id` | `15551` |
| `stop_sequence` | Integer | **Required** | Order of stop in trip (starts at 1) | `1` |
| `stop_headsign` | Text | Optional | Headsign displayed to passengers at this stop | `Downtown` |
| `pickup_type` | Enum | Optional | `0` = regular, `1` = none, `2` = phone, `3` = driver | `0` |
| `drop_off_type` | Enum | Optional | `0` = regular, `1` = none, `2` = phone, `3` = driver | `0` |
| `shape_dist_traveled` | Float | Optional | Distance along shape | `1250.5` |
| `timepoint` | Enum | Optional | `0` = approx, `1` = exact (see `timepoints.txt`) | `1` |


### stops.txt
**Purpose:** All physical bus stops, metro stations, cable car stops, etc.

| Field | Type | Required | Description | Example |
|-------|------|----------|-------------|---------|
| `stop_id` | ID | **Required** | Unique identifier for this stop | `15551` |
| `stop_code` | Text | Optional | Short code displayed to riders | `15551` |
| `stop_name` | Text | **Required** | Name of the stop | `California St & Polk St` |
| `stop_desc` | Text | Optional | Additional description | `NE corner` |
| `stop_lat` | Latitude | **Required** | Latitude (WGS84) | `37.7912` |
| `stop_lon` | Longitude | **Required** | Longitude (WGS84) | `-122.4204` |
| `zone_id` | ID | Optional | Fare zone identifier | `` |
| `stop_url` | URL | Optional | Web page for this stop | `` |
| `location_type` | Enum | Optional | `0` = stop, `1` = station, `2` = entrance | `0` |
| `parent_station` | ID | Optional | Parent station (for grouped stops) | `` |
| `stop_timezone` | Timezone | Optional | Timezone if different from agency | `` |
| `wheelchair_boarding` | Enum | Optional | `0` = no info, `1` = accessible, `2` = not accessible | `1` |
| `level_id` | ID | Optional | Level (floor) within a station | `` |
| `platform_code` | Text | Optional | Platform identifier | `A` |


### timepoints.txt
**Purpose:** *Non-standard extension.* Flags which stops are schedule adherence checkpoints.

| Field | Type | Required | Description | Example |
|-------|------|----------|-------------|---------|
| `trip_id` | ID | **Required** | References `trips.trip_id` | `1000001` |
| `stop_id` | ID | **Required** | References `stops.stop_id` | `15551` |
| `timepoint` | Enum | **Required** | `1` = this stop is a timepoint, `0` = approximate | `1` |

**What is a Timepoint?**
A stop where the vehicle must not depart early where the driver must wait until scheduled time. However, Non-timepoints can depart early if running ahead.


### trips.txt
**Purpose:** Defines individual vehicle runs along routes.

| Field | Type | Required | Description | Example |
|-------|------|----------|-------------|---------|
| `route_id` | ID | **Required** | References `routes.route_id` | `1` |
| `service_id` | ID | **Required** | References `calendar.service_id` | `WKDY` |
| `trip_id` | ID | **Required** | Unique identifier for this trip | `1000001` |
| `trip_headsign` | Text | Optional | Destination sign shown on vehicle | `Downtown` |
| `trip_short_name` | Text | Optional | Short trip identifier (rare) | `` |
| `direction_id` | Enum (0/1) | Optional | `0` = outbound, `1` = inbound (agency-defined) | `0` |
| `block_id` | ID | Optional | Block assignment (same vehicle across trips) | `B1234` |
| `shape_id` | ID | Optional | References `shapes.shape_id` | `1_0_var1` |
| `wheelchair_accessible` | Enum | Optional | `0` = no info, `1` = accessible, `2` = not | `1` |
| `bikes_allowed` | Enum | Optional | `0` = no info, `1` = allowed, `2` = not allowed | `1` |


In [25]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
from collections import Counter
import warnings
import os

# Plot Style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print("Libraries Imported")

Libraries Imported


In [26]:
# Base File Path: Needs to be updated for running on different machines.
BASE_DIR = r"C:\Users\Owner\Documents\GitHub\GTFS-Exploration\GTFS Data Exploration\San Francisco\sf_gtfs_da.ipynb"

In [27]:
# File Paths
paths = {
    "agency"         : os.path.join(BASE_DIR, "agency.txt"),
    "calendar"       : os.path.join(BASE_DIR, "calendar.txt"),
    "calendar_dates" : os.path.join(BASE_DIR, "calendar_dates.txt"),
    "directions"     : os.path.join(BASE_DIR, "directions.txt"),
    "fare_attributes": os.path.join(BASE_DIR, "fare_attributes.txt"),
    "fare_rules"     : os.path.join(BASE_DIR, "fare_rules.txt"),
    "routes"         : os.path.join(BASE_DIR, "routes.txt"),
    "shapes"         : os.path.join(BASE_DIR, "shapes.txt"),
    "stop_times"     : os.path.join(BASE_DIR, "stop_times.txt"),
    "stops"          : os.path.join(BASE_DIR, "stops.txt"),
    "timepoints"     : os.path.join(BASE_DIR, "timepoints.txt"),
    "trips"          : os.path.join(BASE_DIR, "trips.txt"),
}

```
┌─────────────┐
│  agency.txt │
└──────┬──────┘
       │
       ├──────────────┐
       │              │
  ┌────▼─────┐   ┌───▼──────────┐
  │ routes   │   │ calendar     │
  │ .txt     │   │ .txt         │
  └────┬─────┘   └───┬──────────┘
       │             │
       │             │  ┌──────────────────┐
       │             └──► calendar_dates   │
       │                │ .txt             │
       │                └──────────────────┘
       │
  ┌────▼─────┐
  │ trips    │──────┐
  │ .txt     │      │
  └────┬─────┘      │
       │            │
       │     ┌──────▼──────┐
       │     │  shapes.txt │
       │     │  (optional) │
       │     └─────────────┘
       │
  ┌────▼──────────┐
  │  stop_times   │─────┐
  │  .txt         │     │
  └───────────────┘     │
                        │
                   ┌────▼─────┐
                   │ stops    │
                   │ .txt     │
                   └──────────┘
```

**Key Joins:**
- `trips.route_id` → `routes.route_id`
- `trips.service_id` → `calendar.service_id`
- `trips.shape_id` → `shapes.shape_id`
- `stop_times.trip_id` → `trips.trip_id`
- `stop_times.stop_id` → `stops.stop_id`
- `fare_rules.route_id` → `routes.route_id`
- `fare_rules.fare_id` → `fare_attributes.fare_id`

In [15]:
# Code